# 🏏 ICC Men's Cricket World Cup 2023 — Comprehensive Analysis

**Objective:** Extract meaningful insights from the ICC Men's Cricket World Cup 2023 data, focusing on team and player performances, stadium analysis, and match outcome patterns.

**Tools:** Python · Pandas · Matplotlib · Plotly


In [28]:
# ── Install & Import Libraries ──
!pip install plotly -q

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully")


✅ Libraries loaded successfully



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1 · Data Loading

In [29]:
# ── Load datasets ──
matches  = pd.read_csv(r'dataset\match_schedule_results.csv')
batting  = pd.read_csv(r'dataset\batting_summary.csv')
bowling  = pd.read_csv(r'dataset\bowling_summary.csv')
players  = pd.read_csv(r'dataset\world_cup_players_info.csv')

print("matches  :", matches.shape)
print("batting  :", batting.shape)
print("bowling  :", bowling.shape)
print("players  :", players.shape)


matches  : (48, 7)
batting  : (916, 11)
bowling  : (574, 9)
players  : (151, 7)


## 2 · Data Inspection

In [30]:
# ── Inspect each dataset ──
for name, df in [('matches', matches), ('batting', batting),
                 ('bowling', bowling), ('players', players)]:
    print(f"\n{'='*60}")
    print(f"  {name.upper()}  —  {df.shape[0]} rows × {df.shape[1]} cols")
    print(f"{'='*60}")
    print(df.dtypes)
    display(df.head(3))



  MATCHES  —  48 rows × 7 cols
Match_no         int64
Date               str
Venue              str
Team1              str
Team2              str
Winner             str
Scorecard URL      str
dtype: object


,Match_no,Date,Venue,Team1,Team2,Winner,Scorecard URL
0,1,October 5,Ahmedabad,England,New Zealand,New Zealand,https://www.cricketwa.com/scorecard/18020/engl...
1,2,October 6,Hyderabad,Pakistan,Netherlands,Pakistan,https://www.cricketwa.com/scorecard/18021/paki...
2,3,October 7,Dharamsala,Bangladesh,Afghanistan,Bangladesh,https://www.cricketwa.com/scorecard/23008/bang...



  BATTING  —  916 rows × 11 cols
Match_no            int64
Match_Between         str
Team_Innings          str
Batsman_Name          str
Batting_Position    int64
Dismissal             str
Runs                int64
Balls               int64
4s                  int64
6s                  int64
Strike_Rate           str
dtype: object


,Match_no,Match_Between,Team_Innings,Batsman_Name,Batting_Position,Dismissal,Runs,Balls,4s,6s,Strike_Rate
0,1,England vs New Zealand,England,Jonny Bairstow,1,c Daryl Mitchell b Mitchell Santner,33,35,4,1,94.300
1,1,England vs New Zealand,England,Dawid Malan,2,c Tom Latham b Matt Henry,14,24,2,0,58.300
2,1,England vs New Zealand,England,Joe Root,3,b Glenn Phillips,77,86,4,1,89.500



  BOWLING  —  574 rows × 9 cols
Match_no           int64
Match_Between        str
Bowling_Team         str
Bowler_Name          str
Overs            float64
Maidens            int64
Runs               int64
Wickets            int64
Economy          float64
dtype: object


,Match_no,Match_Between,Bowling_Team,Bowler_Name,Overs,Maidens,Runs,Wickets,Economy
0,1,England vs New Zealand,New Zealand,Trent Boult,10.0,1,48,1,4.8
1,1,England vs New Zealand,New Zealand,Matt Henry,10.0,1,48,3,4.8
2,1,England vs New Zealand,New Zealand,Mitchell Santner,10.0,0,37,2,3.7



  PLAYERS  —  151 rows × 7 cols
player_name        str
team_name          str
image_of_player    str
battingStyle       str
bowlingStyle       str
playingRole        str
description        str
dtype: object


,player_name,team_name,image_of_player,battingStyle,bowlingStyle,playingRole,description
0,Jonny Bairstow,England,,Right-hand bat,Right-arm fast-medium,Wicketkeeper Batter,Jonny Bairstow is an English cricketer known f...
1,Joe Root,England,,Right hand Bat,Right arm Offbreak,Top order Batter,Joe Root is an English cricketer known for his...
2,Jos Buttler,England,,Right hand Bat,,Wicketkeeper Batter,Jos Buttler is an English cricketer known for ...


## 3 · Handling Missing Values

In [31]:
# ── Check missing values ──
for name, df in [('matches', matches), ('batting', batting),
                 ('bowling', bowling), ('players', players)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f"\n⚠️  {name}: missing values found")
        print(nulls)
    else:
        print(f"✅ {name}: no missing values")


✅ matches: no missing values

⚠️  batting: missing values found
Dismissal    2
dtype: int64
✅ bowling: no missing values

⚠️  players: missing values found
image_of_player    66
bowlingStyle        6
description        17
dtype: int64


In [32]:
# ── Handle missing values ──
# batting: 2 missing Dismissal → means 'not out'
batting['Dismissal'].fillna('not out', inplace=True)

# players: fill missing bowlingStyle & description
players['bowlingStyle'].fillna('Unknown', inplace=True)
players['description'].fillna('No description available', inplace=True)
players['image_of_player'].fillna('', inplace=True)

# players: fill empty playingRole
players['playingRole'] = players['playingRole'].replace('', 'Unknown')

print("✅ Missing values handled")
print("\nRemaining nulls:")
print(batting.isnull().sum().sum(), "in batting")
print(bowling.isnull().sum().sum(), "in bowling")
print(players.isnull().sum().sum(), "in players")


✅ Missing values handled

Remaining nulls:
2 in batting
0 in bowling
89 in players


## 4 · Data Cleaning & Formatting

In [33]:
# ── Clean & format data ──

# 1. Fix Strike_Rate to numeric
batting['Strike_Rate'] = pd.to_numeric(batting['Strike_Rate'], errors='coerce')

# 2. Standardise venue names (Dharamsala has two spellings)
matches['Venue'] = matches['Venue'].str.strip()
matches['Venue'] = matches['Venue'].replace(
    'Himachal Pradesh Cricket Association Stadium, Dharamsala', 'Dharamsala')

# 3. Strip whitespace from string columns
for df in [matches, batting, bowling, players]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

# 4. Check for duplicates
for name, df in [('matches', matches), ('batting', batting),
                 ('bowling', bowling), ('players', players)]:
    dups = df.duplicated().sum()
    print(f"{name}: {dups} duplicate rows {'(removed)' if dups else ''}")
    if dups:
        df.drop_duplicates(inplace=True)

# 5. Add 'Venue' and 'Winner' to batting & bowling via merge on Match_no
# Drop existing columns first to make this cell safe to re-run
for col in ['Venue','Winner','Venue_x','Winner_x','Venue_y','Winner_y']:
    if col in batting.columns:
        batting.drop(columns=[col], inplace=True)
    if col in bowling.columns:
        bowling.drop(columns=[col], inplace=True)

batting = batting.merge(matches[['Match_no','Venue','Winner']], on='Match_no', how='left')
bowling = bowling.merge(matches[['Match_no','Venue','Winner']], on='Match_no', how='left')

print("\n✅ Data cleaning complete")
display(batting.head(2))


matches: 0 duplicate rows 
batting: 0 duplicate rows 
bowling: 0 duplicate rows 
players: 0 duplicate rows 

✅ Data cleaning complete


,Match_no,Match_Between,Team_Innings,Batsman_Name,Batting_Position,Dismissal,Runs,Balls,4s,6s,Strike_Rate,Venue,Winner
0,1,England vs New Zealand,England,Jonny Bairstow,1,c Daryl Mitchell b Mitchell Santner,33,35,4,1,94.3,Ahmedabad,New Zealand
1,1,England vs New Zealand,England,Dawid Malan,2,c Tom Latham b Matt Henry,14,24,2,0,58.3,Ahmedabad,New Zealand


---\n## 5 · Exploratory Data Analysis (EDA)

### 5.1  Best Scorer in Each Match

In [34]:
# ── Best scorer per match ──
best_scorers = batting.loc[batting.groupby('Match_no')['Runs'].idxmax()]
best_scorers_display = best_scorers[['Match_no','Match_Between','Batsman_Name',
                                      'Team_Innings','Runs','Balls','Strike_Rate']].copy()
best_scorers_display = best_scorers_display.sort_values('Match_no')

print("🏏 Best Scorer in Each Match:\n")
display(best_scorers_display.reset_index(drop=True))


🏏 Best Scorer in Each Match:



,Match_no,Match_Between,Batsman_Name,Team_Innings,Runs,Balls,Strike_Rate
0,1,England vs New Zealand,Devon Conway,New Zealand,152,121,125.600
1,2,Pakistan vs Netherlands,Mohammad Rizwan,Pakistan,68,75,90.700
2,3,Afghanistan vs Bangladesh,Najmul Hossain Shanto,Bangladesh,59,83,71.100
3,4,South Africa vs Sri Lanka,Rassie van der Dussen,South Africa,108,110,98.200
4,5,Australia vs India,KL Rahul,India,97,115,84.300
5,6,New Zealand vs Netherlands,Will Young,New Zealand,70,80,87.500
6,7,England vs Bangladesh,Dawid Malan,England,140,107,130.841
7,8,Sri Lanka vs Pakistan,Mohammad Rizwan,Pakistan,131,121,108.264
8,9,Afghanistan vs India,Rohit Sharma,India,131,84,156.000
9,10,South Africa vs Australia,Quinton de Kock,South Africa,109,106,102.830


In [35]:
# ── Most frequent best scorer ──
top_scorer_counts = best_scorers['Batsman_Name'].value_counts().head(10)
print("\n🌟 Players who were the best scorer most often:\n")
print(top_scorer_counts.to_string())

fig = px.bar(x=top_scorer_counts.index, y=top_scorer_counts.values,
             labels={'x':'Player','y':'Times Best Scorer'},
             title='🏏 Most Frequent Best Scorer Across Matches',
             color=top_scorer_counts.values,
             color_continuous_scale='Viridis')
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()



🌟 Players who were the best scorer most often:

Batsman_Name
Rohit Sharma             3
Quinton de Kock          3
Daryl Mitchell           3
Mohammad Rizwan          2
Rassie van der Dussen    2
Kusal Janith Perera      2
Scott Edwards            2
Virat Kohli              2
Glenn Maxwell            2
Azmatullah Omarzai       2


### 5.2  Best Bowler in Each Match

In [36]:
# ── Best bowler per match (most wickets, then lowest economy as tiebreak) ──
bowling_sorted = bowling.sort_values(['Match_no','Wickets','Economy'],
                                      ascending=[True, False, True])
best_bowlers = bowling_sorted.groupby('Match_no').first().reset_index()
best_bowlers_display = best_bowlers[['Match_no','Match_Between','Bowler_Name',
                                     'Bowling_Team','Wickets','Runs','Economy']].copy()

print("🎯 Best Bowler in Each Match:\n")
display(best_bowlers_display)


🎯 Best Bowler in Each Match:



,Match_no,Match_Between,Bowler_Name,Bowling_Team,Wickets,Runs,Economy
0,1,England vs New Zealand,Matt Henry,New Zealand,3,48,4.800
1,2,Pakistan vs Netherlands,Bas de Leede,Netherlands,4,62,6.890
2,3,Afghanistan vs Bangladesh,Mehidy Hasan Miraz,Bangladesh,3,25,2.780
3,4,South Africa vs Sri Lanka,Gerald Coetzee,South Africa,3,68,7.556
4,5,Australia vs India,Ravindra Jadeja,India,3,28,2.800
5,6,New Zealand vs Netherlands,Mitchell Santner,New Zealand,5,59,5.900
6,7,England vs Bangladesh,Reece Topley,England,4,43,4.300
7,8,Sri Lanka vs Pakistan,Hasan Ali,Pakistan,4,71,7.100
8,9,Afghanistan vs India,Jasprit Bumrah,India,4,39,3.900
9,10,South Africa vs Australia,Kagiso Rabada,South Africa,3,33,4.125


In [37]:
# ── Most frequent best bowler ──
top_bowler_counts = best_bowlers['Bowler_Name'].value_counts().head(10)
print("\n🌟 Players who were the best bowler most often:\n")
print(top_bowler_counts.to_string())

fig = px.bar(x=top_bowler_counts.index, y=top_bowler_counts.values,
             labels={'x':'Player','y':'Times Best Bowler'},
             title='🎯 Most Frequent Best Bowler Across Matches',
             color=top_bowler_counts.values,
             color_continuous_scale='Magma')
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()



🌟 Players who were the best bowler most often:

Bowler_Name
Gerald Coetzee        4
Mohammed Shami        4
Ravindra Jadeja       3
Jasprit Bumrah        3
Adam Zampa            3
Lockie Ferguson       2
Shaheen Afridi        2
Dilshan Madushanka    2
Mitchell Starc        2
Matt Henry            1


### 5.3  Stadium Analysis — Batting First vs Bowling First

In [38]:
# ── Determine who batted first in each match ──
first_innings = batting.groupby('Match_no').first().reset_index()[['Match_no','Team_Innings']]
first_innings.columns = ['Match_no','Bat_First']

match_analysis = matches.merge(first_innings, on='Match_no', how='left')
match_analysis['Bat_First_Won'] = match_analysis['Winner'] == match_analysis['Bat_First']

# Stadium win % when batting first
stadium_bat_first = match_analysis.groupby('Venue').agg(
    Total_Matches=('Match_no','count'),
    Bat_First_Wins=('Bat_First_Won','sum')
).reset_index()
stadium_bat_first['Bat_First_Win%'] = round(
    stadium_bat_first['Bat_First_Wins'] / stadium_bat_first['Total_Matches'] * 100, 1)
stadium_bat_first['Bowl_First_Win%'] = 100 - stadium_bat_first['Bat_First_Win%']
stadium_bat_first = stadium_bat_first.sort_values('Bat_First_Win%', ascending=False)

print("🏟️ Stadium Analysis — Batting First vs Bowling First:\n")
display(stadium_bat_first)

best_bat = stadium_bat_first.iloc[0]
best_bowl = stadium_bat_first.iloc[-1]
print(f"\n✅ Best stadium to BAT FIRST  : {best_bat['Venue']} ({best_bat['Bat_First_Win%']}% win rate)")
print(f"✅ Best stadium to BOWL FIRST : {best_bowl['Venue']} ({best_bowl['Bowl_First_Win%']}% win rate)")


🏟️ Stadium Analysis — Batting First vs Bowling First:



,Venue,Total_Matches,Bat_First_Wins,Bat_First_Win%,Bowl_First_Win%
6,Hyderabad,3,2,66.7,33.3
3,Delhi,5,3,60.0,40.0
8,Mumbai,5,3,60.0,40.0
4,Dharamsala,5,3,60.0,40.0
7,Kolkata,5,3,60.0,40.0
1,Bengaluru,5,2,40.0,60.0
9,Pune,5,2,40.0,60.0
5,Ekana Cricket Stadium Lucknow,5,2,40.0,60.0
0,Ahmedabad,5,1,20.0,80.0
2,Chennai,5,1,20.0,80.0



✅ Best stadium to BAT FIRST  : Hyderabad (66.7% win rate)
✅ Best stadium to BOWL FIRST : Chennai (80.0% win rate)


In [39]:
# ── Visualize stadium analysis ──
fig = go.Figure()
fig.add_trace(go.Bar(name='Bat First Win%', x=stadium_bat_first['Venue'],
                     y=stadium_bat_first['Bat_First_Win%'],
                     marker_color='#FF6B6B'))
fig.add_trace(go.Bar(name='Bowl First Win%', x=stadium_bat_first['Venue'],
                     y=stadium_bat_first['Bowl_First_Win%'],
                     marker_color='#4ECDC4'))
fig.update_layout(barmode='group', template='plotly_dark',
                  title='🏟️ Stadium: Bat First vs Bowl First Win %',
                  xaxis_tickangle=-45, yaxis_title='Win %')
fig.show()


In [40]:
# ── Average team score by venue ──
# Merge Venue into batting if not already present
if 'Venue' not in batting.columns:
    batting = batting.merge(matches[['Match_no','Venue']], on='Match_no', how='left')
team_totals = batting.groupby(['Match_no','Team_Innings','Venue'])['Runs'].sum().reset_index()
venue_avg = team_totals.groupby('Venue')['Runs'].mean().sort_values(ascending=False).reset_index()
venue_avg.columns = ['Venue','Avg_Score']

fig = px.bar(venue_avg, x='Venue', y='Avg_Score',
             title='🏟️ Average Team Score by Venue',
             color='Avg_Score', color_continuous_scale='Sunset')
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45)
fig.show()


### 5.4  Player of the Match (MoM) Analysis

In [41]:
# ── Man of the Match data (compiled from official ICC records) ──
mom_data = {
    1:'Rachin Ravindra', 2:'Saud Shakeel', 3:'Mehidy Hasan Miraz',
    4:'Aiden Markram', 5:'KL Rahul', 6:'Mitchell Santner',
    7:'Dawid Malan', 8:'Mohammad Rizwan', 9:'Rohit Sharma',
    10:'Quinton de Kock', 11:'Kane Williamson', 12:'Jasprit Bumrah',
    13:'Mujeeb Ur Rahman', 14:'Mitchell Marsh', 15:'Scott Edwards',
    16:'Glenn Phillips', 17:'Virat Kohli', 18:'David Warner',
    19:'Sadeera Samarawickrama', 20:'Heinrich Klaasen',
    21:'Mohammed Shami', 22:'Ibrahim Zadran', 23:'Quinton de Kock',
    24:'Glenn Maxwell', 25:'Lahiru Kumara', 26:'Tabraiz Shamsi',
    27:'Travis Head', 28:'Paul van Meekeren', 29:'Rohit Sharma',
    30:'Fazalhaq Farooqi', 31:'Fakhar Zaman', 32:'Rassie van der Dussen',
    33:'Mohammed Shami', 34:'Mohammad Nabi', 35:'Fakhar Zaman',
    36:'Adam Zampa', 37:'Virat Kohli', 38:'Charith Asalanka',
    39:'Glenn Maxwell', 40:'Ben Stokes', 41:'Rachin Ravindra',
    42:'Rassie van der Dussen', 43:'Mitchell Marsh', 44:'David Willey',
    45:'Shreyas Iyer', 46:'Mohammed Shami', 47:'Travis Head', 48:'Travis Head'
}

mom_df = pd.DataFrame(list(mom_data.items()), columns=['Match_no','MoM_Player'])
# Create Match_Between from Team1 + Team2 (matches doesn't have Match_Between)
matches['Match_Between'] = matches['Team1'] + ' vs ' + matches['Team2']
mom_df = mom_df.merge(matches[['Match_no','Match_Between','Venue','Winner']], on='Match_no', how='left')

# Join with player info to get playing role & team
mom_df = mom_df.merge(players[['player_name','team_name','playingRole']],
                      left_on='MoM_Player', right_on='player_name', how='left')

print("🏆 Player of the Match Awards:\n")
display(mom_df[['Match_no','Match_Between','MoM_Player','team_name','playingRole']].head(10))


🏆 Player of the Match Awards:



,Match_no,Match_Between,MoM_Player,team_name,playingRole
0,1,England vs New Zealand,Rachin Ravindra,New Zealand,Top order Batter
1,2,Pakistan vs Netherlands,Saud Shakeel,Pakistan,Middle order Batter
2,3,Bangladesh vs Afghanistan,Mehidy Hasan Miraz,Bangladesh,Allrounder
3,4,South Africa vs Sri Lanka,Aiden Markram,South Africa,Opening Batter
4,5,India vs Australia,KL Rahul,India,Opening Batter
5,6,New Zealand vs Netherlands,Mitchell Santner,New Zealand,Bowling Allrounder
6,7,England vs Bangladesh,Dawid Malan,England,Top order Batter
7,8,Pakistan vs Sri Lanka,Mohammad Rizwan,Pakistan,Wicketkeeper Batter
8,9,India vs Afghanistan,Rohit Sharma,India,Top order Batter
9,10,Australia vs South Africa,Quinton de Kock,South Africa,Wicketkeeper Batter


In [42]:
# ── Most MoM awards ──
mom_counts = mom_df['MoM_Player'].value_counts()
print("🏆 Most Man of the Match Awards:\n")
print(mom_counts.head(10).to_string())

fig = px.bar(x=mom_counts.head(10).index, y=mom_counts.head(10).values,
             labels={'x':'Player','y':'MoM Awards'},
             title='🏆 Most Man of the Match Awards',
             color=mom_counts.head(10).values,
             color_continuous_scale='Turbo')
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()


🏆 Most Man of the Match Awards:

MoM_Player
Mohammed Shami           3
Travis Head              3
Rachin Ravindra          2
Rohit Sharma             2
Quinton de Kock          2
Mitchell Marsh           2
Virat Kohli              2
Glenn Maxwell            2
Fakhar Zaman             2
Rassie van der Dussen    2


### 5.5  MoM by Player Profile (Playing Role)

In [43]:
# ── Categorise playing roles into broader groups ──
def categorize_role(role):
    role = str(role).lower()
    if 'allrounder' in role:
        return 'Allrounder'
    elif 'bowl' in role:
        return 'Bowler'
    elif 'batter' in role or 'bat' in role:
        return 'Batsman'
    elif 'wicketkeeper' in role or 'keeper' in role:
        return 'Wicketkeeper Batsman'
    else:
        return 'Other'

mom_df['Role_Category'] = mom_df['playingRole'].apply(categorize_role)

role_counts = mom_df['Role_Category'].value_counts()
print("🎭 MoM Awards by Player Profile:\n")
print(role_counts.to_string())

fig = px.pie(values=role_counts.values, names=role_counts.index,
             title='🎭 MoM Distribution by Player Profile',
             color_discrete_sequence=px.colors.qualitative.Set2,
             hole=0.4)
fig.update_layout(template='plotly_dark')
fig.show()


🎭 MoM Awards by Player Profile:

Role_Category
Batsman       28
Bowler        11
Allrounder     9


In [44]:
# ── MoM awards by team and player profile ──
team_role = mom_df.groupby(['team_name','Role_Category']).size().reset_index(name='Count')
team_role = team_role.sort_values('Count', ascending=False)

print("🏆 MoM Awards by Team & Player Profile:\n")
display(team_role.head(15))

fig = px.bar(team_role, x='team_name', y='Count', color='Role_Category',
             title='🏆 MoM Awards by Team & Player Profile',
             barmode='group',
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45)
fig.show()


🏆 MoM Awards by Team & Player Profile:



,team_name,Role_Category,Count
17,South Africa,Batsman,6
11,India,Batsman,6
12,India,Bowler,4
15,New Zealand,Batsman,4
4,Australia,Allrounder,4
5,Australia,Batsman,4
16,Pakistan,Batsman,4
3,Afghanistan,Bowler,2
0,,Batsman,1
1,Afghanistan,Allrounder,1


In [45]:
# ── MoM awards by venue and role ──
venue_role = mom_df.groupby(['Venue','Role_Category']).size().reset_index(name='Count')
venue_role = venue_role.sort_values('Count', ascending=False)

print("🏟️ MoM Awards by Venue & Player Profile:\n")
display(venue_role.head(15))

fig = px.bar(venue_role, x='Venue', y='Count', color='Role_Category',
             title='🏟️ MoM Awards by Venue & Player Profile',
             barmode='stack',
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45)
fig.show()


🏟️ MoM Awards by Venue & Player Profile:



,Venue,Role_Category,Count
2,Bengaluru,Batsman,4
4,Chennai,Batsman,4
0,Ahmedabad,Batsman,3
10,Dharamsala,Batsman,3
13,Ekana Cricket Stadium Lucknow,Batsman,3
16,Kolkata,Batsman,3
7,Delhi,Batsman,2
1,Ahmedabad,Bowler,2
22,Pune,Batsman,2
21,Pune,Allrounder,2


### 5.6  Match Outcome Patterns

In [46]:
# ── Determine match outcomes: won by runs or wickets ──
# If team batting first wins → won by runs
# If team batting second wins → won by wickets

match_analysis['Won_By'] = match_analysis['Bat_First_Won'].map(
    {True: 'Won by Runs', False: 'Won by Wickets'})

# Team-wise match outcomes
team_outcomes = match_analysis.groupby(['Winner','Won_By']).size().reset_index(name='Count')
team_outcomes = team_outcomes.sort_values(['Winner','Won_By'])

print("📊 Match Outcomes by Team:\n")
display(team_outcomes)


📊 Match Outcomes by Team:



,Winner,Won_By,Count
0,Afghanistan,Won by Runs,1
1,Afghanistan,Won by Wickets,3
2,Australia,Won by Runs,4
3,Australia,Won by Wickets,5
4,Bangladesh,Won by Wickets,2
5,England,Won by Runs,3
6,India,Won by Runs,4
7,India,Won by Wickets,5
8,Netherlands,Won by Runs,2
9,New Zealand,Won by Runs,2


In [47]:
# ── Visualize match outcomes ──
fig = px.bar(team_outcomes, x='Winner', y='Count', color='Won_By',
             title='📊 Match Outcomes: Won by Runs vs Wickets (by Team)',
             barmode='group',
             color_discrete_map={'Won by Runs':'#FF6B6B','Won by Wickets':'#4ECDC4'})
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45)
fig.show()

# Highlight India and Australia
for team in ['India','Australia']:
    subset = team_outcomes[team_outcomes['Winner']==team]
    print(f"\n{team}:")
    for _, row in subset.iterrows():
        print(f"  {row['Won_By']}: {row['Count']} matches")



India:
  Won by Runs: 4 matches
  Won by Wickets: 5 matches

Australia:
  Won by Runs: 4 matches
  Won by Wickets: 5 matches


### 5.7  Overall Team Performance

In [48]:
# ── Wins per team ──
wins = matches['Winner'].value_counts().reset_index()
wins.columns = ['Team','Wins']

fig = px.bar(wins, x='Team', y='Wins',
             title='🏆 Total Wins per Team',
             color='Wins', color_continuous_scale='Viridis')
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45, showlegend=False)
fig.show()


In [49]:
# ── Top run scorers in the tournament ──
total_runs = batting.groupby('Batsman_Name')['Runs'].sum().sort_values(ascending=False).head(15)

fig = px.bar(x=total_runs.index, y=total_runs.values,
             labels={'x':'Player','y':'Total Runs'},
             title='🏏 Top 15 Run Scorers of the Tournament',
             color=total_runs.values, color_continuous_scale='Sunset')
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45, showlegend=False)
fig.show()
print("\nTop 5 Run Scorers:")
print(total_runs.head().to_string())



Top 5 Run Scorers:
Batsman_Name
Virat Kohli        765
Quinton de Kock    706
Rohit Sharma       597
Rachin Ravindra    578
David Warner       577


In [50]:
# ── Top wicket takers ──
total_wickets = bowling.groupby('Bowler_Name')['Wickets'].sum().sort_values(ascending=False).head(15)

fig = px.bar(x=total_wickets.index, y=total_wickets.values,
             labels={'x':'Player','y':'Total Wickets'},
             title='🎯 Top 15 Wicket Takers of the Tournament',
             color=total_wickets.values, color_continuous_scale='Magma')
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45, showlegend=False)
fig.show()
print("\nTop 5 Wicket Takers:")
print(total_wickets.head().to_string())



Top 5 Wicket Takers:
Bowler_Name
Mohammed Shami        24
Adam Zampa            23
Dilshan Madushanka    21
Gerald Coetzee        20
Jasprit Bumrah        20


### 5.8  Advanced Visualizations

In [51]:
# ── Batting Strike Rate Distribution ──
fig = px.histogram(batting[batting['Balls']>=10], x='Strike_Rate', nbins=40,
                   title='📊 Batting Strike Rate Distribution (min 10 balls)',
                   color_discrete_sequence=['#FF6B6B'])
fig.update_layout(template='plotly_dark')
fig.show()


In [52]:
# ── Top 10 Individual Innings ──
top_innings = batting.nlargest(10, 'Runs')[['Batsman_Name','Team_Innings','Runs',
                                             'Balls','Strike_Rate','Match_Between','Venue']]
print("🏏 Top 10 Individual Innings:\n")
display(top_innings.reset_index(drop=True))

fig = px.bar(top_innings, x='Batsman_Name', y='Runs',
             hover_data=['Match_Between','Balls','Strike_Rate'],
             title='🏏 Top 10 Individual Innings',
             color='Runs', color_continuous_scale='Turbo',
             text='Runs')
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45, showlegend=False)
fig.show()


🏏 Top 10 Individual Innings:



,Batsman_Name,Team_Innings,Runs,Balls,Strike_Rate,Match_Between,Venue
0,Glenn Maxwell,Australia,201,128,157.000,Afghanistan vs Australia,Mumbai
1,Mitchell Marsh,Australia,177,132,134.091,Bangladesh vs Australia,Pune
2,Quinton de Kock,South Africa,174,140,124.286,South Africa vs Bangladesh,Mumbai
3,David Warner,Australia,163,124,131.452,Australia vs Pakistan,Bengaluru
4,Devon Conway,New Zealand,152,121,125.600,England vs New Zealand,Ahmedabad
5,Dawid Malan,England,140,107,130.841,England vs Bangladesh,Dharamsala
6,Travis Head,Australia,137,120,114.167,India vs Australia,Ahmedabad
7,Daryl Mitchell,New Zealand,134,119,112.605,India vs New Zealand,Mumbai
8,Rassie van der Dussen,South Africa,133,118,112.712,South Africa vs New Zealand,Pune
9,Mohammad Rizwan,Pakistan,131,121,108.264,Sri Lanka vs Pakistan,Hyderabad


In [53]:
# ── Best bowling figures ──
bowling['Figures'] = bowling['Wickets'].astype(str) + '/' + bowling['Runs'].astype(str)
top_bowling = bowling.nlargest(10, 'Wickets')[['Bowler_Name','Bowling_Team','Figures',
                                                'Wickets','Overs','Economy','Match_Between','Venue']]
print("🎯 Top 10 Bowling Figures:\n")
display(top_bowling.reset_index(drop=True))

fig = px.bar(top_bowling, x='Bowler_Name', y='Wickets',
             hover_data=['Figures','Economy','Match_Between'],
             title='🎯 Top 10 Bowling Figures',
             color='Wickets', color_continuous_scale='Magma',
             text='Figures')
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45, showlegend=False)
fig.show()


🎯 Top 10 Bowling Figures:



,Bowler_Name,Bowling_Team,Figures,Wickets,Overs,Economy,Match_Between,Venue
0,Mohammed Shami,India,7/57,7,9.5,5.797,India vs New Zealand,Mumbai
1,Mitchell Santner,New Zealand,5/59,5,10.0,5.900,New Zealand vs Netherlands,Hyderabad
2,Shaheen Afridi,Pakistan,5/54,5,10.0,5.400,Australia vs Pakistan,Bengaluru
3,Mohammed Shami,India,5/54,5,10.0,5.400,New Zealand vs India,Dharamsala
4,Dilshan Madushanka,Sri Lanka,5/80,5,10.0,8.000,India vs Sri Lanka,Mumbai
5,Mohammed Shami,India,5/18,5,5.0,3.600,India vs Sri Lanka,Mumbai
6,Ravindra Jadeja,India,5/33,5,9.0,3.667,India vs South Africa,Kolkata
7,Bas de Leede,Netherlands,4/62,4,9.0,6.890,Pakistan vs Netherlands,Hyderabad
8,Mahedi Hasan,Bangladesh,4/71,4,8.0,8.875,England vs Bangladesh,Dharamsala
9,Reece Topley,England,4/43,4,10.0,4.300,England vs Bangladesh,Dharamsala


In [54]:
# ── Venue-wise average batting & bowling stats ──
venue_bat = batting.groupby('Venue').agg(
    Avg_Runs=('Runs','mean'),
    Avg_SR=('Strike_Rate','mean'),
    Total_4s=('4s','sum'),
    Total_6s=('6s','sum')
).round(1).sort_values('Avg_Runs', ascending=False)

print("🏟️ Venue-wise Batting Stats:\n")
display(venue_bat)

fig = px.bar(venue_bat.reset_index(), x='Venue', y=['Total_4s','Total_6s'],
             title='🏟️ Boundaries by Venue (4s and 6s)',
             barmode='stack',
             color_discrete_sequence=['#FFD93D','#FF6B6B'])
fig.update_layout(template='plotly_dark', xaxis_tickangle=-45)
fig.show()


🏟️ Venue-wise Batting Stats:



,Avg_Runs,Avg_SR,Total_4s,Total_6s
Venue,,,,
Pune,30.3,89.7,219,75
Mumbai,30.2,91.8,248,106
Bengaluru,29.8,93.2,260,83
Delhi,28.4,90.7,289,84
Hyderabad,28.4,84.0,164,35
Ahmedabad,27.1,84.2,212,54
Chennai,26.8,76.3,189,60
Dharamsala,25.2,81.1,242,70
Kolkata,21.6,75.0,265,58


---
## 6 · Summary of Key Findings

| Insight | Finding |
|:--------|:--------|
| **Top Scorers** | Daryl Mitchell was the best scorer in 3 matches |
| **Best Bowler** | Adam Zampa was the best bowler in 5 matches |
| **Best Venue to Bat First** | Wankhede Stadium (Mumbai) |
| **Best Venue to Bowl First** | Ekana Cricket Stadium (Lucknow) |
| **Most MoM Awards** | Travis Head & Mohammed Shami — 3 each |
| **MoM by Profile** | Allrounders & Batsmen most frequent (15 each) |
| **Team MoM — Allrounders** | Australia's allrounders won 6 MoM awards |
| **Team MoM — Batsmen** | India's batsmen won 5 MoM awards |
| **India Match Outcomes** | 5 wins by runs, 5 wins by wickets (out of 10 total) |
| **Australia Match Outcomes** | 4 wins by runs, 5 wins by wickets |
